# Clase 2 — De una tabla a una decisión

**Laboratorio de Modelo Analítico para la Toma de Decisiones**
ISTEA · 2026-2C · División A

---

### Cómo usar este notebook

Se lee de arriba hacia abajo. Las celdas grises son código: hacés clic adentro y
apretás **Shift + Enter** para ejecutarlas. Todo lo demás es texto para leer.

**No hace falta saber pandas.** Cada vez que aparezca algo nuevo te lo explico antes
de usarlo. Si algo no se entiende, preguntá en el momento — es la clase 2, nadie
tiene por qué saber esto todavía.

Vamos a hacer una sola cosa, y la vas a ver completa: **convertir una tabla de
compras en una tabla de personas**, y descubrir algo que antes no se podía decir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


---

## Antes del código: qué es un modelo analítico

Esta parte no se ejecuta, se lee. Son dos minutos y es la idea sobre la que se apoya
toda la materia.

### Un reporte y un modelo no son lo mismo

Un **reporte** dice qué pasó:

> «Facturamos $ 5.500 millones.»
> «La categoría Audio creció 12 %.»
> «Tenemos 4.000 clientes.»

Todo eso es verdad y es útil. Pero fijate que ninguna de esas frases le dice a nadie
qué hacer mañana a la mañana.

Un **modelo analítico** hace otra cosa. Toma lo que pasó y produce **una decisión
sobre un caso concreto**:

> «A este cliente, llamalo esta semana.»
> «A este otro, no gastes: se va igual.»

|  | Reporte | Modelo analítico |
| --- | --- | --- |
| ¿De qué habla? | De la empresa, del mes, de la categoría | **De una persona** |
| ¿Qué responde? | Qué pasó | Qué conviene hacer con *esta* |
| ¿Cuándo sirve? | Después | Antes |
| ¿Qué devuelve? | Un número | Una acción, y por qué |

### El movimiento que hace la diferencia

Para pasar de una cosa a la otra hay que hacer algo que suena menor y no lo es:
**cambiar de qué habla cada fila de la tabla.**

Hoy vas a ver ese movimiento. Es una línea de código y cambia todo lo que se puede
preguntar después.

### Dónde estamos parados

La materia son cuatro pasos, y cada uno es una unidad:

| Paso | Qué se hace | Clases |
| --- | --- | --- |
| 1 · Relevar | Entender qué datos hay y si alcanzan | 2 a 5 |
| 2 · Construir la tabla del cliente | Pasar de una fila por compra a una por persona | 6 a 8 |
| **3 · Modelar** | Encontrar qué separa al que se queda del que se va | 10 a 13 |
| 4 · Decidir y medir | Traducir eso en una acción con plata | 14 a 16 |

Hoy hacemos el **paso 2 en miniatura**, para que se vea a dónde vamos. No te
preocupes si no entendés cada línea: hoy alcanza con ver el resultado.

---

## 1 · Primero, que la máquina funcione

Antes que nada tenemos que confirmar dos cosas: que Colab te corre código y que
tenés disponible la herramienta que vamos a usar.

**pandas** es una biblioteca de Python para trabajar con tablas. Ya viene instalada
en Colab, así que solo hay que traerla con `import`. Se la suele abreviar `pd` por
costumbre — vas a ver ese `pd` en todos lados de acá en adelante.

In [ ]:
import pandas as pd

print("pandas versión", pd.__version__)
print("Listo: Colab funciona.")

pandas versión 2.2.3
Listo: Colab funciona.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Qué tenés que ver:** dos líneas, una con un número de versión (algo como `2.3.3`)
y otra que dice *Listo*.

Si en vez de eso ves texto rojo, avisá y lo miramos ahora.

---

## 2 · Traer los datos

> ### Antes de correr la celda de abajo
> En el panel de la izquierda, hacé clic en el **ícono de carpeta** 📁 y después en
> el botón de **subir archivo**. Subí `eleccommsphere_transacciones.csv`.
>
> Si no ves el panel, apretá el ícono de la carpeta en la barra lateral izquierda.

### Con qué vamos a trabajar

**ElecCommSphere** es el caso de la materia: una plataforma argentina de comercio
electrónico. Su buscador viene devolviendo resultados que no tienen nada que ver con
lo que la gente busca, y en el área de Retención sospechan que por eso se están
yendo clientes. La Dirección quiere evidencia antes de aprobar plata para arreglarlo.

Ese es el caso que voy a usar yo durante toda la cursada para mostrarles cada
técnica. **El de ustedes va a ser otro**, real, y lo eligen hoy.

### Qué hace la línea que sigue

`pd.read_csv()` abre un archivo `.csv` y lo convierte en una **tabla** que Python
puede manipular. En pandas esa tabla se llama **DataFrame**: pensalo como una hoja
de Excel, pero adentro del código.

Lo guardamos en una variable que llamamos `tx`, de *transacciones*.

In [ ]:
tx = pd.read_csv("/content/drive/MyDrive/eleccommsphere_transacciones.csv", parse_dates=["fecha"])

print("La tabla tiene", tx.shape[0], "filas y", tx.shape[1], "columnas.")

La tabla tiene 99441 filas y 5 columnas.


**Qué tenés que ver:** *La tabla tiene 26310 filas y 12 columnas.*

Si sale `FileNotFoundError`, es que el archivo no está subido. Volvé al panel de la
izquierda.

### Dos cosas de esa línea

`parse_dates=["fecha"]` le avisa a pandas que la columna `fecha` es una fecha y no
un texto. Parece un detalle y no lo es: si entra como texto, después no se puede
ordenar por fecha, ni restar dos fechas, ni agrupar por mes. Es el error más común de
la materia.

`tx.shape` devuelve dos números: cuántas filas y cuántas columnas tiene la tabla.

### Ahora miremos la tabla

`.head()` muestra las primeras cinco filas. Sirve para ver con qué estás trabajando
antes de hacer cualquier cosa.

In [ ]:
tx.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


### Frená un segundo acá. Esto es lo importante de toda la clase.

Mirá bien la tabla que acaba de salir.

**Cada fila es una compra.** No un cliente: una compra. El mismo cliente puede
aparecer cien veces, o una sola. Fijate en la columna `id_cliente`: hay valores que
se repiten.

Ahora escuchá la pregunta que quiere responder el negocio:

> ### ¿A qué clientes estamos perdiendo?

¿Ves el problema?

**La pregunta es sobre personas. La tabla es sobre compras.** No hay forma de
responderla con la tabla así como está. No importa cuánto sepas de Python: primero
hay que cambiar de qué habla cada fila.

Eso es lo que vamos a hacer en el punto 4.

---

## 3 · Antes de reorganizar: los cuatro roles

> Esta es la idea que más te va a servir en todo el cuatrimestre. Si te llevás una
> sola cosa de hoy, que sea esta.

Todo lo que vamos a hacer de acá a noviembre —recencia, frecuencia, monto,
segmentación, predicción— se apoya en **cuatro roles**.

Ojo con la palabra: **roles**, no nombres de columna. En cada dataset se llaman
distinto, pero cumplen la misma función.

| Rol | Qué tiene que ser | En ElecCommSphere se llama |
| --- | --- | --- |
| **1 · Un identificador que se repite** | Quién hizo la operación. Tiene que aparecer muchas veces. | `id_cliente` |
| **2 · Una fecha** | Cuándo pasó. Sirve para separar pasado de futuro. | `fecha` |
| **3 · Un valor sumable** | Cuánto vale. Es lo que está en juego. | `monto` |
| **4 · Algo que salió mal** *(opcional)* | Una devolución, un reclamo, un error. | `devuelto` |

### Por qué el primero es el más importante

Si el identificador **no se repite** —si cada cliente aparece una sola vez— no hay
historia que mirar. No se puede saber si alguien compra seguido o si dejó de comprar,
porque solo tenemos una foto de cada persona.

Sin repetición no hay nada que predecir. Es el filtro que más datasets descarta.

### Ahora en código

En la celda de abajo guardamos los nombres de las columnas en tres variables. Fijate
que están en mayúsculas: es una convención para marcar «esto es una configuración,
no un dato».

**Estas tres líneas son las únicas que cambian cuando pasás a otro dataset.**

In [ ]:
ID    = "id_cliente"     # rol 1: el identificador que se repite
FECHA = "fecha"          # rol 2: cuándo pasó
VALOR = "monto"          # rol 3: cuánto vale

# tx[ID] agarra una columna entera de la tabla, la de id_cliente.
# .nunique() cuenta cuántos valores DISTINTOS hay adentro.

cantidad_clientes = tx[ID].nunique()

print("Identificador :", ID)
print("   clientes distintos:", cantidad_clientes)
print()
print("Fecha         :", FECHA)
print("   desde", tx[FECHA].min().date(), "hasta", tx[FECHA].max().date())
print()
print("Valor sumable :", VALOR)
print("   total facturado: $", f"{tx[VALOR].sum():,.0f}")

Identificador : id_cliente
   clientes distintos: 3301

Fecha         : fecha
   desde 2025-01-06 hasta 2026-06-30

Valor sumable : monto
   total facturado: $ 5,554,920,247


**Qué tenés que ver:** 3.301 clientes distintos, fechas entre enero de 2025 y junio
de 2026, y un total facturado de unos $ 5.554 millones.

### La cuenta que decide si un dataset sirve

Tenemos 26.310 compras hechas por 3.301 personas. Dividí una cosa por la otra:

In [ ]:
filas_por_cliente = len(tx) / tx[ID].nunique()

print("Compras por cliente, en promedio:", round(filas_por_cliente, 1))

Compras por cliente, en promedio: 8.0


**Qué tenés que ver:** *8.0*

Ocho compras por persona. Hay historia de sobra para mirar.

Si ese número diera **1,0**, querría decir que cada cliente compró una sola vez, y
todo lo que viene después de esta clase sería imposible. Cuando abras tu dataset en
la clase 3, esta es la primera cuenta que vas a hacer.

---

## 4 · El movimiento: de compras a personas

Llegamos. Acá está lo que vinimos a ver.

### Qué es agrupar

Imaginate que tenés las 26.310 compras impresas en papelitos, todas mezcladas sobre
una mesa. **Agrupar** es hacer pilas: una pila por cliente, con todos los papelitos
de esa persona juntos.

Una vez que tenés las pilas, de cada una podés sacar un resumen: cuántos papelitos
tiene, cuánto suman, cuál es el más reciente. Y ese resumen es **una fila nueva**,
una por persona.

En pandas eso se hace con `groupby`:

```python
tx.groupby(ID)     # hacé una pila por cada valor distinto de ID
```

Y después le pedís los resúmenes con `.agg()`, que viene de *agregar*:

```python
.agg(
    nombre_nuevo = (columna_que_mira, qué_le_hace)
)
```

Las operaciones que vamos a usar son cuatro y se leen solas: `size` cuenta cuántas
filas hay en la pila, `sum` las suma, `mean` calcula el promedio y `max` se queda con
el valor más grande.

In [ ]:
clientes = tx.groupby(ID).agg(
    frecuencia = (ID,    "size"),   # cuántas compras hizo
    monto      = (VALOR, "sum"),    # cuánto gastó en total
    ticket     = (VALOR, "mean"),   # cuánto gasta por compra, en promedio
    ultima     = (FECHA, "max"),    # la fecha de su última compra
)

print("Antes:", len(tx),       "filas — cada una es una COMPRA")
print("Ahora:", len(clientes), "filas — cada una es una PERSONA")

clientes.head()

Antes: 26310 filas — cada una es una COMPRA
Ahora: 3301 filas — cada una es una PERSONA


,frecuencia,monto,ticket,ultima
id_cliente,,,,
1,1,75954.37,75954.370000,2026-04-27
3,6,1238677.18,206446.196667,2026-06-08
6,2,365787.22,182893.610000,2026-06-09
7,2,805543.74,402771.870000,2025-09-21
8,2,339613.68,169806.840000,2026-06-07


**Qué tenés que ver:** 26.310 filas se convirtieron en 3.301, y una tabla nueva donde
cada fila tiene un `id_cliente` y cuatro columnas que lo resumen.

**Esa tabla habla de personas.** La anterior hablaba de compras. Eso es todo lo que
cambió, y alcanza para cambiar qué preguntas se pueden hacer.

### Falta una columna, y es la más útil

Sabemos cuándo fue la última compra de cada cliente. Pero lo que interesa no es la
fecha en sí: es **cuánto tiempo pasó desde entonces**. A eso se le dice **recencia**.

Se calcula restando: la fecha más reciente del dataset menos la última compra de esa
persona. Restar dos fechas en pandas da una duración, y `.dt.days` la convierte en un
número de días.

In [ ]:
CORTE = tx[FECHA].max()          # la fecha más reciente de todo el dataset

clientes["recencia"] = (CORTE - clientes["ultima"]).dt.days

print("Fecha de corte:", CORTE.date())
print()
clientes[["frecuencia", "monto", "recencia"]].head()

Fecha de corte: 2026-06-30



,frecuencia,monto,recencia
id_cliente,,,
1,1,75954.37,64
3,6,1238677.18,22
6,2,365787.22,21
7,2,805543.74,282
8,2,339613.68,23


### Lo que acabás de construir tiene nombre

Esas tres columnas —**recencia**, **frecuencia** y **monto**— se llaman **RFM**.

Es una de las herramientas más viejas y más usadas del análisis de clientes: tiene
más de cincuenta años y se aplica igual en un banco, en un supermercado o en una
plataforma de streaming. Y son tres líneas de código.

En la **clase 8** las vas a construir vos, con once variables en vez de tres y
partiendo el tiempo en dos. Hoy alcanza con que veas de dónde salen.

---

## 5 · Para qué sirve tener la tabla así

Ahora que cada fila es una persona, se pueden hacer preguntas que antes no existían.

Probemos una: **¿cuánta gente dejó de comprar?**

Vamos a filtrar la tabla y quedarnos solo con los que hace más de 90 días que no
aparecen. En pandas, filtrar se escribe poniendo una condición entre corchetes.

In [ ]:
dormidos = clientes[clientes["recencia"] > 90]

print("Clientes que no compran hace más de 90 días:")
print("  cantidad:", f"{len(dormidos):,}")
print("  son el  :", f"{len(dormidos) / len(clientes):.1%}", "del total")
print()
print("Facturación histórica que representan:")
print("  $", f"{dormidos['monto'].sum():,.0f}")
print("  o sea el", f"{dormidos['monto'].sum() / clientes['monto'].sum():.1%}",
      "de todo lo que facturó la empresa")

Clientes que no compran hace más de 90 días:
  cantidad: 1,397
  son el  : 42.3% del total

Facturación histórica que representan:
  $ 980,985,996
  o sea el 17.7% de todo lo que facturó la empresa


### Leé despacio lo que dice ese resultado

**Cuatro de cada diez clientes de esta empresa no compran hace más de tres meses. Y
entre todos representan casi el 18 % de todo lo que la empresa facturó en su
historia.**

Fijate una cosa: no agregamos ni un dato nuevo. Es exactamente la misma información
que estaba en el archivo cuando lo abrimos. Lo único que hicimos fue **reorganizarla
alrededor del cliente en vez de alrededor de la compra**.

Un número así se lleva a una reunión y alguien decide algo con él. Y todo lo que
hicimos fue un `groupby` y una resta.

---

## 6 · La prueba de que esto no era un truco

Todo lo que hicimos usó tres nombres de columna: `id_cliente`, `fecha`, `monto`.

En tu dataset esas columnas se llaman distinto. **El código es el mismo.**

Abajo está el bloque de carga de los trece datasets del catálogo. Buscá el tuyo,
sacale los `#` del principio de esas líneas y ejecutá la celda. Después **volvé a la
celda del `groupby` y corrés todo de nuevo, sin cambiar nada más**.

Lo único que se toca son las tres variables `ID`, `FECHA` y `VALOR`.

> Esto lo podés hacer ahora si te da el tiempo, o en tu casa esta semana. Para la
> clase 3 tenés que llegar con tu archivo abierto.

In [ ]:
# ============ BUSCÁ TU DATASET, SACALE LOS # Y EJECUTÁ ============

# ---------- Online Retail II ----------
# !wget -q -O r.zip "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip" && unzip -o -q r.zip
# hojas = pd.read_excel("online_retail_II.xlsx", sheet_name=None)
# tx = pd.concat(hojas.values(), ignore_index=True)
# tx["monto"] = tx["Quantity"] * tx["Price"]
# ID, FECHA, VALOR = "Customer ID", "InvoiceDate", "monto"

# ---------- MovieLens 25M ----------
# !wget -q -O ml.zip https://files.grouplens.org/datasets/movielens/ml-25m.zip && unzip -o -q ml.zip
# tx = pd.read_csv("ml-25m/ratings.csv")
# tx["fecha"] = pd.to_datetime(tx["timestamp"], unit="s")
# ID, FECHA, VALOR = "userId", "fecha", "rating"

# ---------- Amazon (cambiá CAT por la categoría que te tocó) ----------
# CAT = "Video_Games"
# opciones: Video_Games, Musical_Instruments, Software, CDs_and_Vinyl,
#           Office_Products, Grocery_and_Gourmet_Food, Toys_and_Games
# URL = ("https://mcauleylab.ucsd.edu/public_datasets/data"
#        "/amazon_2023/benchmark/5core/rating_only/" + CAT + ".csv.gz")
# tx = pd.read_csv(URL)
# tx["fecha"] = pd.to_datetime(tx["timestamp"], unit="ms")
# ID, FECHA, VALOR = "user_id", "fecha", "rating"

# ---------- Los cuatro de Kaggle ----------
# Estos piden que subas tu archivo kaggle.json (Settings -> API -> Create New Token)
# !pip -q install kaggle && mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Complete Journey:
# !kaggle datasets download -d frtgnn/dunnhumby-the-complete-journey -q --unzip -p dh
# tx = pd.read_csv("dh/transaction_data.csv")
# tx["fecha"] = pd.to_datetime(tx["DAY"], unit="D", origin="2016-01-01")
# ID, FECHA, VALOR = "household_key", "fecha", "SALES_VALUE"

# Olist — ojo: acá el cliente es el VENDEDOR, no el comprador
!kaggle datasets download -d olistbr/brazilian-ecommerce -q --unzip -p ol
items   = pd.read_csv("ol/olist_order_items_dataset.csv")
ordenes = pd.read_csv("ol/olist_orders_dataset.csv", parse_dates=["order_purchase_timestamp"])
tx = items.merge(ordenes, on="order_id")
ID, FECHA, VALOR = "seller_id", "order_purchase_timestamp", "price"

# Superstore:
# !kaggle datasets download -d vivek468/superstore-dataset-final -q --unzip -p ss
# tx = pd.read_csv("ss/Sample - Superstore.csv", encoding="latin-1", parse_dates=["Order Date"])
# ID, FECHA, VALOR = "Customer ID", "Order Date", "Sales"

# Sparkov:
# !kaggle datasets download -d kartik2112/fraud-detection -q --unzip -p sp
# tx = pd.read_csv("sp/fraudTrain.csv", parse_dates=["trans_date_trans_time"])
# ID, FECHA, VALOR = "cc_num", "trans_date_trans_time", "amt"

# ==================================================================

print("Configuración actual:")
print("   ID    =", ID)
print("   FECHA =", FECHA)
print("   VALOR =", VALOR)
print()
print("Ahora volvé a la celda del groupby y ejecutala. No toques nada más.")

Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
License(s): CC-BY-NC-SA-4.0
Configuración actual:
   ID    = seller_id
   FECHA = order_purchase_timestamp
   VALOR = price

Ahora volvé a la celda del groupby y ejecutala. No toques nada más.


---

## 7 · Lo que hicimos hoy todavía no es un modelo

Y conviene ser honesto con eso.

Lo de hoy **describe**: dice cuántos clientes están dormidos y cuánta plata
representan. Pero no **decide** nada. Un modelo diría *«a este cliente llamalo, a
este no»*, y para eso faltan tres cosas.

| Qué falta | Cuándo lo vemos | Por qué hace falta |
| --- | --- | --- |
| **Limpiar los datos** | Clases 3 a 5 | Corrimos sobre datos sucios: hay compras duplicadas, montos vacíos y clientes que no existen en el sistema. Los números de hoy están mal a propósito, y en la clase 4 los vas a arreglar. |
| **Partir el tiempo** | Clase 8 | Usamos todo el período de una. Para predecir hay que mirar solo una parte y dejar la otra tapada, como si el futuro todavía no hubiera pasado. Si no, te estás haciendo trampa. |
| **Un modelo y un umbral** | Clases 12 y 13 | Saber quién está en riesgo no alcanza. Hay que decidir a cuántos contactar, y eso depende de cuánto cuesta contactar y cuánto se gana si el cliente se queda. |

Lo de hoy es el esqueleto. En noviembre, esto mismo que corriste en una celda va a
ser un documento que recomienda una acción concreta y dice cuánta plata hay en juego.

---

### Si algo no funcionó

| Lo que dice el error | Qué pasó | Cómo se arregla |
| --- | --- | --- |
| `FileNotFoundError` | El CSV no está subido a Colab | Panel izquierdo → ícono de carpeta → subir archivo |
| `NameError: name 'tx' is not defined` | No ejecutaste una celda anterior | Volvé arriba y ejecutá todo en orden |
| `NameError: name 'pd' is not defined` | No ejecutaste el `import pandas` | Andá a la primera celda de código |
| `KeyError: 'monto'` | Esa columna no existe en tu dataset | Ejecutá `tx.columns` y fijate cómo se llama de verdad |

Si aparece otra cosa, copiá el error y preguntá. Los errores en Python son largos
pero la última línea suele decir bastante.